In [3]:
import pandas as pd
import nltk

In [1]:
text1 = open('./no_titles/OZina_1_Lij.txt', 'r', encoding='utf-8').read()
text2 = open('./no_titles/OZina_2_Lij.txt', 'r', encoding='utf-8').read()
text3 = open('./no_titles/OZina_3_Lij.txt', 'r', encoding='utf-8').read()
final = text1 + '\n\n' + text2 + '\n\n' + text3

In [6]:
nltk.download('punkt')
sentences = nltk.sent_tokenize(final)
rows = []
for sentence in sentences:
	tokens = nltk.word_tokenize(sentence)
	for word in tokens:
		rows.append({
			'SENTENCE': sentence,
			'TEXT': word,
			'POS_GROUPED': None
		})

df_gen = pd.DataFrame(rows)
df_gen.head()
df_gen.to_csv('postagging_genoese_SENTENCES.csv', index=False)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Sara\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [7]:
import pandas as pd

# 1. Carica i file
df_pos = pd.read_csv('postagging_genoese.csv')
df_sent = pd.read_csv('postagging_genoese_SENTENCES.csv')

# 2. Crea una colonna 'original_order' per non perdere la sequenza delle frasi
df_sent['original_order'] = range(len(df_sent))

# 3. Pulizia: rimuovi la colonna POS_GROUPED vuota se presente
if 'POS_GROUPED' in df_sent.columns:
    df_sent = df_sent.drop(columns=['POS_GROUPED'])

# 4. Aggiungi 'word_instance' per gestire le parole ripetute (es. "o", "pe", ",")
# Questo assicura che il primo "o" della frase si colleghi al primo "o" dei tag
df_pos['word_instance'] = df_pos.groupby('TEXT').cumcount()
df_sent['word_instance'] = df_sent.groupby('TEXT').cumcount()

# 5. Unisci usando un 'left join' mantenendo df_sent come base
df_final = pd.merge(df_sent, df_pos, on=['TEXT', 'word_instance'], how='left')

# 6. ORDINA il risultato in base all'indice originale e rimuovi le colonne tecniche
df_final = df_final.sort_values('original_order')
df_final = df_final.drop(columns=['word_instance', 'original_order'])

# Salva il risultato ordinato
df_final.to_csv('postagging_genoese_ORDINATO.csv', index=False)

df_final.head()

,SENTENCE,TEXT,POS_GROUPED
0,"O Zouhaur Atif, o zoeno arrestou pe avei ammas...",O,PRON
1,"O Zouhaur Atif, o zoeno arrestou pe avei ammas...",Zouhaur,NOUN
2,"O Zouhaur Atif, o zoeno arrestou pe avei ammas...",Atif,NOUN
3,"O Zouhaur Atif, o zoeno arrestou pe avei ammas...",",",PUNCT
4,"O Zouhaur Atif, o zoeno arrestou pe avei ammas...",o,PRON
